In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import torch
import matplotlib.pyplot as plt

# Load data

In [3]:
ice_vel = xr.load_dataset("/home/kim/data/nsidc/antarctic_ice_vel_phase_map_v01.nc")

# Define new variable: speed V: sqrt(VX**2 + VY**2)
# hypot is numerically more stable than sqrt(VX**2 + VY**2)
ice_vel["V"] = np.hypot(ice_vel["VX"], ice_vel["VY"])

In [4]:
# import region bounds
from regions import ROSS_BOUNDS

# assign region bounds
x_min = ROSS_BOUNDS["x_min"]
x_max = ROSS_BOUNDS["x_max"]
y_min = ROSS_BOUNDS["y_min"]
y_max = ROSS_BOUNDS["y_max"]

ice_vel_ross = ice_vel.sel(
    x = slice(x_min, x_max), 
    y = slice(y_max, y_min))

ice_vel_ross

<xarray.Dataset> Size: 242MB
Dimensions:       (x: 2223, y: 2222)
Coordinates:
  * x             (x) float64 18kB -6e+05 -5.995e+05 ... 3.995e+05 4e+05
  * y             (y) float64 18kB -4.004e+05 -4.008e+05 ... -1.399e+06 -1.4e+06
    lat           (y, x) float64 40MB -83.37 -83.37 -83.38 ... -76.66 -76.66
    lon           (y, x) float64 40MB 236.3 236.3 236.2 ... 164.1 164.1 164.1
Data variables:
    coord_system  |S1 1B b''
    VX            (y, x) float32 20MB 12.16 12.11 11.48 10.77 ... nan nan nan
    VY            (y, x) float32 20MB 0.08527 0.174 0.1077 ... nan nan nan
    STDX          (y, x) float32 20MB 0.2943 0.1676 0.2148 ... nan nan nan
    STDY          (y, x) float32 20MB 0.7608 0.4851 0.683 1.092 ... nan nan nan
    ERRX          (y, x) float32 20MB 0.4791 0.3609 0.3734 ... nan nan nan
    ERRY          (y, x) float32 20MB 0.2238 0.1654 0.2066 ... nan nan nan
    CNT           (y, x) int32 20MB 15 15 15 15 15 15 15 15 ... 0 0 0 0 0 0 0 0
    SOURCE        (y, x) int8 5MB 3 3 3 3 3 3 3 3 3 1 1 ... 0 0 0 0 0 0 0 0 0 0
    V             (y, x) float32 20MB 12.16 12.11 11.48 10.77 ... nan nan nan
Attributes: (12/27)
    Conventions:               CF-1.6
    Metadata_Conventions:      CF-1.6, Unidata Dataset Discovery v1.0, GDS v2.0
    standard_name_vocabulary:  CF Standard Name Table (v22, 12 February 2013)
    id:                        v_mix.v8Jul2019.nc
    title:                     MEaSURES Antarctica Ice Velocity Map 450m spacing
    product_version:            
    ...                        ...
    time_coverage_start:       1995-01-01
    time_coverage_end:         2016-12-31
    project:                   NASA/MEaSUREs
    creator_name:              J. Mouginot
    comment:                    
    license:                   No restrictions on access or use.

# Interpolate it  on target grid

In [6]:
tighter_ice_shelf_mask = xr.load_dataarray("data/tighter_ice_shelf_mask.nc")

mask_for_ice_vel = tighter_ice_shelf_mask.interp(
    x = ice_vel_ross.x, 
    y = ice_vel_ross.y, 
    method = "nearest"
)

ice_vel_ross_on_target = ice_vel_ross.V.where(mask_for_ice_vel == 3)

# Load MODIS background

In [7]:
modis_ross = torch.load("data/modis/moa125_2014_hp1_v01_ross_with_grid_no_ocean.pt", weights_only = False)

# Visualise

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))

# 1) MODIS as background (draw first)
ax.pcolormesh(
    modis_ross[0],
    modis_ross[1], 
    modis_ross[2],
    cmap = "gray",
    # softer greys
    vmin = -30_000, 
    vmax = 30_000,
    # as less saturated background
    alpha = 0.4,
    zorder = 0,
)

# 2) SMB on top (draw second)
pcm = ax.pcolormesh(
    ice_vel_ross_on_target.x,
    ice_vel_ross_on_target.y,
    ice_vel_ross_on_target,
    cmap = "viridis",
    # Fix these limits for consistency
    vmin = 0,
    vmax = 1200,
    shading = "auto",
    # 1.0 for full opacity & consistency
    alpha = 1.0,
    zorder = 1,
)

ax.set_aspect("equal")
ax.set_axis_off()

fig.savefig("figures/ice_vel_ross.png", dpi = 300, bbox_inches = "tight", pad_inches = 0)
plt.show()

# cmap

In [8]:
cmap = "viridis"
dpi = 300

# 1 x N gradient
grad = np.linspace(0, 1, 2048)[None, :]

fig = plt.figure(figsize = (6, 0.6), dpi = dpi)
ax = fig.add_axes([0, 0, 1, 1]) 
ax.imshow(grad, aspect = "auto", cmap = cmap)
ax.set_axis_off()

fig.savefig("figures/cmap/viridis_bar.png", dpi = dpi, transparent = True, bbox_inches = "tight", pad_inches = 0)
plt.close(fig)

# Create directional guidance training data

In [62]:
# Interpolate to target grid
ice_vel_on_target = ice_vel_ross[["VX", "VY"]].interp(
    x = tighter_ice_shelf_mask.x, 
    y = tighter_ice_shelf_mask.y,
    method = "nearest"
)

In [ ]:
X, Y = xr.broadcast(ice_vel_on_target.x, ice_vel_on_target.y)
# Mask it
X_col = X.values.flatten()[~ np.isnan(tighter_ice_shelf_mask.values.flatten())]
Y_col = Y.values.flatten()[~ np.isnan(tighter_ice_shelf_mask.values.flatten())]
VX_col = ice_vel_on_target.VX.values.flatten()[~ np.isnan(tighter_ice_shelf_mask.values.flatten())]
VY_col = ice_vel_on_target.VY.values.flatten()[~ np.isnan(tighter_ice_shelf_mask.values.flatten())]

# Normalize to [0, 1]
X_col_norm = (X_col - x_min) / (x_max - x_min)
Y_col_norm = (Y_col - y_min) / (y_max - y_min)

# Make unit vectors (more stable for loss)
V_magnitudes = np.hypot(VX_col, VY_col)
VX_unit = VX_col / V_magnitudes
VY_unit = VY_col / V_magnitudes

# Save as torch tensor
velocity_unit_norm = torch.tensor(
    np.stack([X_col_norm, Y_col_norm, VX_unit, VY_unit], axis = 1), 
    dtype = torch.float32
)

# Remove incomplete rows
mask_missing_values = ~ torch.isnan(velocity_unit_norm).any(dim=1)
velocity_unit_norm = velocity_unit_norm[mask_missing_values]

print(velocity_unit_norm.shape)

torch.save(velocity_unit_norm, "data/directional_guidance/velocity_unit_norm.pt")

torch.Size([3353602, 4])


In [69]:
velocity_unit_norm

tensor([[ 7.5000e-04,  9.9975e-01,  1.0000e+00,  1.5316e-03],
        [ 7.5000e-04,  9.9925e-01,  9.9999e-01, -5.1309e-03],
        [ 7.5000e-04,  9.9875e-01,  9.9971e-01, -2.3926e-02],
        ...,
        [ 9.6225e-01,  7.7750e-02,  4.9615e-01, -8.6824e-01],
        [ 9.6275e-01,  8.3750e-02, -2.4940e-01, -9.6840e-01],
        [ 9.6275e-01,  8.3250e-02, -1.5459e-01, -9.8798e-01]])